In [1]:
#!/usr/bin/env python3
"""
bayes_missing.py -- joint Bayesian logistic regression with missing covariates
and per-variable selection models for (possibly non-ignorable) missingness.

Pure numpy/scipy Gibbs sampler (no PyMC/Stan needed).

Model
-----
Base variables z_i (q, never missing), vitals x_i (p, partly missing), outcome y_i.

 1. Vitals  : x_i | z_i ~ MVN(B' [1, z_i], Sigma)              B ~ matrix-normal, Sigma ~ IW
 2. Outcome : y_i | z_i, x_i ~ Bernoulli(logit^-1(a + z_i b_z + x_i b_x))
              (Polya-gamma augmentation -> all conditionals conjugate)
 3. Missingness (only for vitals you list in `selection`; others are treated as ignorable)
              R_ij = 1[observed],  P(R_ij = 1) = Phi(g0 + z_i g_z + y_i g_y + g_x * x_ij)
              (Albert-Chib probit augmentation)

    * g_x is the NON-IGNORABLE part: missingness depends on the vital's own value.
      selection[name] = None   -> g_x free, prior N(0, tau_gx^2)
      selection[name] = delta  -> g_x FIXED at delta (pattern-mixture-style sensitivity analysis)
    * vitals not in `selection` are assumed MCAR/MAR given (z, y, other observed values):
      the likelihood factorises and no missingness model is needed.

Missing x_ij are latent parameters: each Gibbs sweep draws them from their full
conditional, which combines (i) the MVN vitals model, (ii) the outcome likelihood and
(iii) the selection model.  So imputation uncertainty propagates into every credible
interval, and the outcome informs the imputations (congenial imputation).
"""
from __future__ import annotations

import os

for _v in ("OMP_NUM_THREADS", "OPENBLAS_NUM_THREADS", "MKL_NUM_THREADS"):
    os.environ.setdefault(_v, "1")

import time
from concurrent.futures import ProcessPoolExecutor

import numpy as np
import pandas as pd
from scipy import linalg
from scipy.special import log_ndtr, ndtri, ndtri_exp
from scipy.stats import invwishart


# --------------------------------------------------------------------------- samplers
def sample_pg1(c, rng, K=200):
    """Draws from Polya-gamma PG(1, c): truncated series + mean-corrected remainder."""
    c = np.abs(c)
    k = np.arange(1, K + 1) - 0.5
    denom = k[None, :] ** 2 + (c[:, None] ** 2) / (4 * np.pi ** 2)
    g = rng.standard_gamma(1.0, size=denom.shape)
    s = (g / denom).sum(1) / (2 * np.pi ** 2)
    full_mean = np.where(c > 1e-6, np.tanh(c / 2) / (2 * np.maximum(c, 1e-6)), 0.25)
    trunc_mean = (1.0 / denom).sum(1) / (2 * np.pi ** 2)
    return s + (full_mean - trunc_mean)


def draw_latent_probit(mean, observed, rng):
    """w ~ N(mean, 1) truncated to w>0 where observed else w<0 (numerically stable)."""
    u = np.clip(rng.random(mean.shape), 1e-12, 1.0)
    # R=0: w = m + Phi^-1(u Phi(-m)) ; R=1: w = m - Phi^-1(u Phi(m))
    z0 = ndtri_exp(np.log(u) + log_ndtr(-mean))
    z1 = ndtri_exp(np.log(u) + log_ndtr(mean))
    return np.where(observed, mean - z1, mean + z0)


def _chol_sample(prec, rhs, rng):
    """Draw from N(prec^-1 rhs, prec^-1)."""
    L = np.linalg.cholesky(prec)
    mean = linalg.cho_solve((L, True), rhs)
    eps = rng.standard_normal(len(rhs))
    return mean + linalg.solve_triangular(L, eps, lower=True, trans="T")


# --------------------------------------------------------------------------- data prep
def prepare(df, base_cols, vital_cols, outcome_col, log_vars=()):
    """Standardise base variables (fully observed) and vitals (on observed values).

    Returns a dict; coefficients come back per 1 'model unit' and are mapped to
    original units with `x_scale`.
    """
    base = df[base_cols].to_numpy(float)
    z_loc, z_scale = base.mean(0), base.std(0)
    Z = (base - z_loc) / z_scale
    X = df[vital_cols].to_numpy(float).copy()
    for j, name in enumerate(vital_cols):
        if name in log_vars:
            X[:, j] = np.log(X[:, j])
    x_loc, x_scale = np.nanmean(X, 0), np.nanstd(X, 0)
    X = (X - x_loc) / x_scale
    return dict(Z=Z, X=X, y=df[outcome_col].to_numpy(float), z_scale=z_scale,
                x_loc=x_loc, x_scale=x_scale, base_cols=list(base_cols), vital_cols=list(vital_cols))


# --------------------------------------------------------------------------- one chain
def run_chain(Z, X, y, selection, names, n_iter=3000, n_burn=1000, seed=0,
              tau_beta=2.5, tau_g=2.0, tau_gx=1.0, tau_B=np.sqrt(10.0),
              prior_obs_rate=None, tau_g0_informative=0.5, verbose=False):
    """Gibbs sampler for one chain.  `selection`: {vital_name: None | fixed g_x}."""
    rng = np.random.default_rng(seed)
    N, p = X.shape
    q = Z.shape[1]
    obs = ~np.isnan(X)
    miss = ~obs
    Xc = np.where(obs, X, 0.0)                    # start imputations at the (standardised) mean
    kb = 1 + q + p
    kappa = y - 0.5

    # --- vitals model
    Zc = np.column_stack([np.ones(N), Z])
    K0 = np.eye(q + 1) / tau_B ** 2
    V_B = np.linalg.inv(Zc.T @ Zc + K0)
    L_VB = np.linalg.cholesky(V_B)
    B = V_B @ Zc.T @ Xc
    Sigma = np.cov((Xc - Zc @ B).T) + 1e-3 * np.eye(p)
    nu0, Psi0 = p + 2, np.eye(p)

    # --- outcome model
    beta = np.zeros(kb)
    P0 = np.eye(kb) / tau_beta ** 2

    # --- selection models
    sel = [j for j, nm in enumerate(names) if nm in selection]
    M = np.column_stack([np.ones(N), Z, y])        # [1, z, y]
    km = M.shape[1]
    fixed = {j: selection[names[j]] for j in sel}
    gam = {j: np.zeros(km + 1) for j in sel}
    for j in sel:
        if fixed[j] is not None:
            gam[j][km] = fixed[j]
    w = np.zeros((N, p))
    prior_m = {}
    prior_sd = {}
    for j in sel:
        m0 = np.zeros(km + 1)
        sd = np.full(km + 1, tau_g)
        sd[km] = tau_gx
        if prior_obs_rate is not None and names[j] in prior_obs_rate:       # informative prior on the
            m0[0] = ndtri(np.clip(prior_obs_rate[names[j]], 1e-3, 1 - 1e-3))  # observation rate
            sd[0] = tau_g0_informative
        prior_m[j], prior_sd[j] = m0, sd
        w[:, j] = np.where(obs[:, j], 0.5, -0.5)

    n_keep = n_iter - n_burn
    out = dict(
        beta=np.zeros((n_keep, kb)),
        gamma=np.zeros((n_keep, len(sel), km + 1)),
        imp_mean=np.zeros((n_keep, p)),
        Sigma=np.zeros((n_keep, p, p)),
        B=np.zeros((n_keep, q + 1, p)),
        X_post_mean=np.zeros((N, p)),
        sel=sel,
    )
    t0 = time.time()
    for it in range(n_iter):
        # (1) outcome: Polya-gamma + Gaussian update of (a, b_z, b_x)
        D = np.column_stack([np.ones(N), Z, Xc])
        eta = D @ beta
        om = sample_pg1(eta, rng)
        beta = _chol_sample((D * om[:, None]).T @ D + P0, D.T @ kappa, rng)
        eta = D @ beta

        # (2) selection models: latent probit scores, then coefficients
        for j in sel:
            g = gam[j]
            w[:, j] = draw_latent_probit(M @ g[:km] + g[km] * Xc[:, j], obs[:, j], rng)
            if fixed[j] is None:
                G, target = np.column_stack([M, Xc[:, j]]), w[:, j]
                prec0 = np.diag(1 / prior_sd[j] ** 2)
                m0 = prior_m[j]
            else:
                G, target = M, w[:, j] - fixed[j] * Xc[:, j]
                prec0 = np.diag(1 / prior_sd[j][:km] ** 2)
                m0 = prior_m[j][:km]
            draw = _chol_sample(G.T @ G + prec0, G.T @ target + prec0 @ m0, rng)
            gam[j] = np.append(draw, g[km]) if fixed[j] is not None else draw

        # (3) missing vitals: full conditionals (Gaussian)
        mu = Zc @ B
        Om = np.linalg.inv(Sigma)
        R = Xc - mu
        for j in range(p):
            idx = miss[:, j]
            if not idx.any():
                continue
            Rj = R[idx]
            cond_mean = mu[idx, j] - (Rj @ Om[:, j] - Rj[:, j] * Om[j, j]) / Om[j, j]
            prec = np.full(idx.sum(), Om[j, j])
            lin = Om[j, j] * cond_mean
            bj = beta[1 + q + j]
            eta_minus = eta[idx] - bj * Xc[idx, j]
            prec = prec + om[idx] * bj ** 2
            lin = lin + bj * (kappa[idx] - om[idx] * eta_minus)
            if j in gam:
                gx = gam[j][km]
                prec = prec + gx ** 2
                lin = lin + gx * (w[idx, j] - M[idx] @ gam[j][:km])
            var = 1.0 / prec
            xnew = lin * var + np.sqrt(var) * rng.standard_normal(idx.sum())
            eta[idx] += bj * (xnew - Xc[idx, j])
            Xc[idx, j] = xnew
            R[idx, j] = xnew - mu[idx, j]

        # (4) vitals regression and covariance
        Mn = V_B @ Zc.T @ Xc
        E = Xc - Zc @ Mn
        S = Psi0 + E.T @ E + Mn.T @ K0 @ Mn
        Sigma = np.atleast_2d(invwishart.rvs(df=nu0 + N, scale=S, random_state=rng))
        B = Mn + L_VB @ rng.standard_normal((q + 1, p)) @ np.linalg.cholesky(Sigma).T

        if it >= n_burn:
            k = it - n_burn
            out["beta"][k] = beta
            for a, j in enumerate(sel):
                out["gamma"][k, a] = gam[j]
            out["imp_mean"][k] = [Xc[miss[:, j], j].mean() if miss[:, j].any() else np.nan for j in range(p)]
            out["Sigma"][k] = Sigma
            out["B"][k] = B
            out["X_post_mean"] += Xc / n_keep
        if verbose and (it + 1) % 500 == 0:
            print(f"  [seed {seed}] iter {it + 1}/{n_iter}  {time.time() - t0:.0f}s", flush=True)
    return out


def _run(args):
    return run_chain(**args)


def fit(data, selection=None, chains=2, n_iter=3000, n_burn=1000, seed=1, verbose=True, **kw):
    """Run several chains in parallel. Returns dict with arrays shaped (chains, draws, ...)."""
    selection = selection or {}
    jobs = [dict(Z=data["Z"], X=data["X"], y=data["y"], selection=selection, names=data["vital_cols"],
                 n_iter=n_iter, n_burn=n_burn, seed=seed * 1000 + c, verbose=verbose and c == 0, **kw)
            for c in range(chains)]
    if chains == 1:
        res = [_run(jobs[0])]
    else:
        with ProcessPoolExecutor(max_workers=min(chains, os.cpu_count() or 1)) as ex:
            res = list(ex.map(_run, jobs))
    stack = lambda k: np.stack([r[k] for r in res])
    return dict(beta=stack("beta"), gamma=stack("gamma"), imp_mean=stack("imp_mean"),
                Sigma=stack("Sigma"), B=stack("B"), X_post_mean=np.mean([r["X_post_mean"] for r in res], 0),
                sel=res[0]["sel"], data=data, selection=selection)


# --------------------------------------------------------------------------- diagnostics
def rhat(x):
    """Split R-hat. x: (chains, n)."""
    m, n = x.shape
    h = n // 2
    s = np.concatenate([x[:, :h], x[:, h:2 * h]], 0)
    n_, m_ = s.shape[1], s.shape[0]
    W = s.var(1, ddof=1).mean()
    Bn = s.mean(1).var(ddof=1)
    return float(np.sqrt(((n_ - 1) / n_ * W + Bn) / W)) if W > 0 else np.nan


def ess(x):
    """Effective sample size (Geyer initial positive sequence). x: (chains, n)."""
    m, n = x.shape
    xc = x - x.mean(1, keepdims=True)
    f = np.fft.rfft(xc, 2 * n, axis=1)
    acov = np.fft.irfft(f * np.conj(f), axis=1)[:, :n] / n * (n / (n - 1))
    W = acov[:, 0].mean()
    var_plus = (n - 1) / n * W + (x.mean(1).var(ddof=1) if m > 1 else 0.0)
    if var_plus <= 0:
        return np.nan
    rho = 1 - (W - acov.mean(0)) / var_plus
    tau, t = 0.0, 0
    while t + 1 < n:
        pair = rho[t] + rho[t + 1]
        if pair < 0:
            break
        tau += pair
        t += 2
    tau = max(-1 + 2 * tau, 1e-3)
    return float(m * n / tau)


def summarize(draws, labels, scale=None):
    """draws: (chains, n, k). Optional `scale` (k,) multiplies draws (unit conversion)."""
    d = draws if scale is None else draws * np.asarray(scale)
    flat = d.reshape(-1, d.shape[-1])
    return pd.DataFrame({
        "mean": flat.mean(0), "sd": flat.std(0),
        "q2.5": np.quantile(flat, 0.025, 0), "q97.5": np.quantile(flat, 0.975, 0),
        "rhat": [rhat(draws[:, :, k]) for k in range(d.shape[-1])],
        "ess": [ess(draws[:, :, k]) for k in range(d.shape[-1])],
    }, index=labels)


def coef_labels(res):
    return ["intercept"] + res["data"]["base_cols"] + res["data"]["vital_cols"]


def coef_scale(res):
    """Multiplier converting model-scale slopes to 'per original SD' units (1 for base vars)."""
    q = len(res["data"]["base_cols"])
    return np.concatenate([np.ones(1 + q), res["data"]["x_scale"]])


In [4]:
#!/usr/bin/env python3
"""
run_analysis.py -- fit the joint Bayesian missing-data model to the synthetic EHR data,
compare it with standard approaches, check convergence, and run a sensitivity analysis.

Methods compared (all coefficients reported per 1 SD of the TRUE vital / base variable,
i.e. the scale the truth was generated on):

  oracle        logistic regression on the complete data (before deletion; unattainable)
  complete-case logistic regression on patients with all 11 vitals observed
  mean-imput.   mean imputation + logistic regression
  MICE          chained equations (20 imputations, Rubin's rules), outcome in imputation model
  Bayes-MAR     joint Bayesian model, missingness ignorable (MAR)
  Bayes-MNAR    joint Bayesian model + selection model for every vital (g_x free)

Then a delta-shift sensitivity analysis for the suspected-MNAR vitals.
"""
import os

for _v in ("OMP_NUM_THREADS", "OPENBLAS_NUM_THREADS", "MKL_NUM_THREADS"):
    os.environ.setdefault(_v, "1")

import argparse
import json
import time
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
from scipy.special import expit

import bayes_missing as bm

warnings.filterwarnings("ignore")

LOG_VARS = ("glucose", "lactate", "creatinine", "wbc")
# Vitals a clinician would flag as plausibly MNAR, with the assumed DIRECTION of the effect:
# +1 = higher values more likely to be recorded, -1 = lower values more likely to be recorded.
SUSPECT = {"glucose": +1, "lactate": +1, "creatinine": +1, "sodium": -1}
SENS_GRID = [0.0, 0.25, 0.5, 0.75, 1.0]      # |g_x| per SD, probit scale


# ------------------------------------------------------------------ frequentist helpers
def logit_fit(D, y, iters=100):
    b = np.zeros(D.shape[1])
    for _ in range(iters):
        pr = expit(D @ b)
        H = (D * (pr * (1 - pr))[:, None]).T @ D + 1e-8 * np.eye(D.shape[1])
        step = np.linalg.solve(H, D.T @ (y - pr))
        b += step
        if np.abs(step).max() < 1e-8:
            break
    return b, np.linalg.inv(H)


def freq_table(b, cov, to_true):
    est, se = b[1:] * to_true, np.sqrt(np.diag(cov))[1:] * to_true
    return pd.DataFrame({"est": est, "lo": est - 1.96 * se, "hi": est + 1.96 * se})


def design(Z, X):
    return np.column_stack([np.ones(len(Z)), Z, X])


def mice_fit(Z, X, y, to_true, m=20):
    from sklearn.experimental import enable_iterative_imputer  # noqa: F401
    from sklearn.impute import IterativeImputer
    from sklearn.linear_model import BayesianRidge
    A = np.column_stack([Z, X, y])
    bs, vs = [], []
    for k in range(m):
        imp = IterativeImputer(estimator=BayesianRidge(), sample_posterior=True, max_iter=10, random_state=k)
        Ai = imp.fit_transform(A)
        b, cov = logit_fit(design(Ai[:, :Z.shape[1]], Ai[:, Z.shape[1]:-1]), y)
        bs.append(b)
        vs.append(np.diag(cov))
    bs, vs = np.array(bs), np.array(vs)
    qbar = bs.mean(0)
    T = vs.mean(0) + (1 + 1 / m) * bs.var(0, ddof=1)          # Rubin's rules
    est, se = qbar[1:] * to_true, np.sqrt(T)[1:] * to_true
    return pd.DataFrame({"est": est, "lo": est - 1.96 * se, "hi": est + 1.96 * se})


def bayes_table(res, to_true):
    labels = bm.coef_labels(res)
    s = bm.summarize(res["beta"], labels, scale=np.concatenate([[1.0], to_true]))
    s = s.iloc[1:]
    return s, pd.DataFrame({"est": s["mean"].values, "lo": s["q2.5"].values, "hi": s["q97.5"].values})


def metrics(tab, truth, mnar_idx):
    err = tab["est"].values - truth
    return dict(RMSE_all=float(np.sqrt((err ** 2).mean())),
                RMSE_mnar_vitals=float(np.sqrt((err[mnar_idx] ** 2).mean())),
                coverage=float(((tab["lo"].values <= truth) & (truth <= tab["hi"].values)).mean()),
                mean_width=float((tab["hi"] - tab["lo"]).mean()))


# ------------------------------------------------------------------ plotting
INK, INK2, MUTED, GRID, SURF = "#0b0b0b", "#52514e", "#8a8984", "#e6e5e1", "#fcfcfb"
C_BLUE, C_ORANGE, C_AQUA, C_YELLOW = "#2a78d6", "#eb6834", "#1baf7a", "#eda100"


def _style(ax, title):
    ax.set_facecolor(SURF)
    ax.set_title(title, loc="left", fontsize=11, color=INK, fontweight="bold")
    ax.tick_params(colors=INK2, labelsize=9, length=0)
    for sp in ("top", "right", "left"):
        ax.spines[sp].set_visible(False)
    ax.spines["bottom"].set_color(GRID)
    ax.grid(axis="x", color=GRID, lw=0.8)
    ax.set_axisbelow(True)


def make_recovery_figure(coef, imp, gam, labels, vit, path):
    import matplotlib
    matplotlib.use("Agg")
    import matplotlib.pyplot as plt
    fig, axes = plt.subplots(1, 3, figsize=(18, 7.2), facecolor=SURF,
                             gridspec_kw=dict(width_ratios=[1.25, 1, 1]))
    # A: estimation error
    ax = axes[0]
    _style(ax, "A  Error in outcome coefficients (estimate - truth)")
    meth = [("Complete-case", C_BLUE), ("MICE", C_ORANGE), ("Bayes, MAR", C_AQUA), ("Bayes, MNAR selection", C_YELLOW)]
    off = np.linspace(-0.27, 0.27, 4)
    truth = coef["truth"].values
    for (name, col), o in zip(meth, off):
        t = coef[name]
        y = np.arange(len(labels)) + o
        ax.hlines(y, t["lo"] - truth, t["hi"] - truth, color=col, lw=1.6, alpha=0.9)
        ax.plot(t["est"] - truth, y, "o", color=col, ms=6, mec=SURF, mew=1.5, label=name)
    ax.axvline(0, color=INK, lw=1.2)
    ax.set_yticks(range(len(labels)))
    ax.set_yticklabels(labels, color=INK2)
    ax.invert_yaxis()
    ax.set_xlabel("log-odds per 1 SD (0 = truth)", color=INK2, fontsize=9)
    ax.legend(frameon=False, fontsize=9, loc="upper center", bbox_to_anchor=(0.5, -0.08), ncol=2, labelcolor=INK2)
    # B: imputation check
    ax = axes[1]
    _style(ax, "B  Mean of the missing values: imputed - true")
    y = np.arange(len(vit))
    ax.plot(imp["observed_mean"] - imp["true_missing_mean"], y, "o", mfc=SURF, mec=MUTED, mew=1.6, ms=7,
            label="Observed mean (no adjustment)")
    ax.plot(imp["bayes_mar"] - imp["true_missing_mean"], y, "o", color=C_AQUA, ms=7, mec=SURF, mew=1.5, label="Bayes, MAR")
    ax.plot(imp["bayes_mnar"] - imp["true_missing_mean"], y, "o", color=C_YELLOW, ms=7, mec=SURF, mew=1.5,
            label="Bayes, MNAR selection")
    ax.axvline(0, color=INK, lw=1.2)
    ax.set_yticks(y)
    ax.set_yticklabels([f"{v}  ({m})" for v, m in zip(vit, imp["mech"])], color=INK2, fontsize=8.5)
    ax.invert_yaxis()
    ax.set_xlabel("difference, in SDs of the observed values (0 = unbiased)", color=INK2, fontsize=9)
    ax.legend(frameon=False, fontsize=9, loc="upper center", bbox_to_anchor=(0.5, -0.08), ncol=2, labelcolor=INK2)
    # C: g_x
    ax = axes[2]
    _style(ax, "C  Own-value effect on being observed (g_x)")
    ax.hlines(y, gam["lo"], gam["hi"], color=C_YELLOW, lw=1.8)
    ax.plot(gam["est"], y, "o", color=C_YELLOW, ms=7, mec=SURF, mew=1.5, label="Posterior mean, 95% CrI")
    ax.plot(gam["true"], y, "D", color=INK, ms=6, mec=SURF, mew=1.2, label="Truth")
    ax.axvline(0, color=MUTED, lw=1.0, ls=":")
    ax.set_yticks(y)
    ax.set_yticklabels([])
    ax.invert_yaxis()
    ax.set_xlabel("probit coefficient per 1 SD of the vital (0 = ignorable)", color=INK2, fontsize=9)
    ax.legend(frameon=False, fontsize=9, loc="upper center", bbox_to_anchor=(0.5, -0.08), ncol=2, labelcolor=INK2)
    fig.tight_layout()
    fig.savefig(path, dpi=140, facecolor=SURF)
    plt.close(fig)


def make_sensitivity_figure(sens, truth_by_vital, path):
    import matplotlib
    matplotlib.use("Agg")
    import matplotlib.pyplot as plt
    fig, ax = plt.subplots(figsize=(8.5, 5.2), facecolor=SURF)
    _style(ax, "Sensitivity of outcome coefficients to the assumed MNAR strength")
    ax.grid(axis="y", color=GRID, lw=0.8)
    cols = dict(zip(SUSPECT, [C_BLUE, C_ORANGE, C_AQUA, C_YELLOW]))
    for v, c in cols.items():
        est, lo, hi = sens[f"{v}__est"], sens[f"{v}__lo"], sens[f"{v}__hi"]
        ax.fill_between(sens["s"], lo, hi, color=c, alpha=0.12, lw=0)
        ax.plot(sens["s"], est, "-o", color=c, lw=2, ms=6, mec=SURF, mew=1.5, label=v)
        ax.axhline(truth_by_vital[v], color=c, lw=1, ls="--")
        ax.text(sens["s"].iloc[-1] + 0.02, est.iloc[-1], v, color=INK2, fontsize=9, va="center")
    ax.set_xlabel("assumed |g_x| (probit, per SD) -- higher values more likely recorded (sodium: lower)", color=INK2, fontsize=9)
    ax.set_ylabel("log-odds per 1 SD (dashed = truth)", color=INK2, fontsize=9)
    ax.set_xlim(-0.03, 1.2)
    ax.legend(frameon=False, fontsize=9, loc="upper right", labelcolor=INK2, ncol=2)
    fig.tight_layout()
    fig.savefig(path, dpi=140, facecolor=SURF)
    plt.close(fig)


# ------------------------------------------------------------------ main
def main():
    ap = argparse.ArgumentParser()
    here = Path.cwd()
    if not (here / 'bayes_missing.py').exists() and (here / 'Preprocessing' / 'bayes_missing.py').exists():
        here = here / 'Preprocessing'
    ap.add_argument("--data-dir", default=str(here / "data"))
    ap.add_argument("--out-dir", default=str(here / "results"))
    ap.add_argument("--iter", type=int, default=6000)
    ap.add_argument("--burn", type=int, default=2500)
    ap.add_argument("--chains", type=int, default=2)
    ap.add_argument("--skip-sensitivity", action="store_true")
    a = ap.parse_args([])  # notebook: use defaults (parsing the kernel's argv would fail)
    out = Path(a.out_dir)
    out.mkdir(parents=True, exist_ok=True)

    df = pd.read_csv(Path(a.data_dir) / "synthetic_ehr.csv")
    full = pd.read_csv(Path(a.data_dir) / "synthetic_ehr_complete.csv")
    truth = json.loads((Path(a.data_dir) / "truth.json").read_text())
    base, vit = truth["base"], truth["vitals"]
    q, p = len(base), len(vit)
    spread = np.array(truth["vital_spread"])
    beta_true = np.array(truth["beta_base"] + truth["beta_vitals"])
    labels = base + vit
    mnar_idx = np.array([q + vit.index(v) for v in ("glucose", "lactate", "creatinine", "sodium")])

    data = bm.prepare(df, base, vit, "outcome", LOG_VARS)
    factor = data["x_scale"] / spread                      # model slope -> per true SD
    to_true = np.concatenate([np.ones(q), factor])
    Z, X, y = data["Z"], data["X"], data["y"]
    coef = {"truth": pd.Series(beta_true)}

    print("== baselines", flush=True)
    dfull = bm.prepare(full, base, vit, "outcome", LOG_VARS)
    b, cov = logit_fit(design(dfull["Z"], dfull["X"]), dfull["y"])
    coef["Oracle (complete data)"] = freq_table(b, cov, np.concatenate([np.ones(q), dfull["x_scale"] / spread]))
    cc = ~np.isnan(X).any(1)
    b, cov = logit_fit(design(Z[cc], X[cc]), y[cc])
    coef["Complete-case"] = freq_table(b, cov, to_true)
    print(f"   complete cases: {cc.sum()} of {len(cc)}", flush=True)
    b, cov = logit_fit(design(Z, np.nan_to_num(X)), y)
    coef["Mean imputation"] = freq_table(b, cov, to_true)
    coef["MICE"] = mice_fit(Z, X, y, to_true)

    print("== Bayesian model, MAR (ignorable)", flush=True)
    t0 = time.time()
    res_mar = bm.fit(data, {}, chains=a.chains, n_iter=a.iter, n_burn=a.burn, seed=1)
    diag_mar, coef["Bayes, MAR"] = bayes_table(res_mar, to_true)
    print(f"   {time.time() - t0:.0f}s", flush=True)

    print("== Bayesian model, selection model for every vital (g_x free)", flush=True)
    t0 = time.time()
    res_sel = bm.fit(data, {v: None for v in vit}, chains=a.chains, n_iter=a.iter, n_burn=a.burn, seed=2)
    diag_sel, coef["Bayes, MNAR selection"] = bayes_table(res_sel, to_true)
    print(f"   {time.time() - t0:.0f}s", flush=True)

    # ---- coefficient table
    wide = pd.DataFrame({"truth": beta_true}, index=labels)
    for k, v in coef.items():
        if k == "truth":
            continue
        v = v.copy()
        v.index = labels
        wide[k] = v["est"]
        wide[k + " lo"] = v["lo"]
        wide[k + " hi"] = v["hi"]
    wide.round(3).to_csv(out / "coefficients.csv")
    est_cols = ["truth", "Oracle (complete data)", "Complete-case", "Mean imputation", "MICE",
                "Bayes, MAR", "Bayes, MNAR selection"]
    print("\nOutcome coefficients (log-odds per 1 SD)")
    print(wide[est_cols].round(2).to_string())

    rows = {}
    for k in est_cols[1:]:
        rows[k] = metrics(coef[k], beta_true, mnar_idx)
    mt = pd.DataFrame(rows).T
    mt.round(3).to_csv(out / "metrics.csv")
    print("\nAccuracy against the truth (single simulated dataset)")
    print(mt.round(3).to_string())

    # ---- convergence
    dg = pd.concat([diag_mar.add_prefix("MAR_"), diag_sel.add_prefix("MNAR_")], axis=1)
    dg.round(3).to_csv(out / "convergence_beta.csv")
    print(f"\nConvergence (outcome coefficients): max R-hat  MAR={diag_mar['rhat'].max():.3f}  "
          f"MNAR={diag_sel['rhat'].max():.3f};  min ESS  MAR={diag_mar['ess'].min():.0f}  MNAR={diag_sel['ess'].min():.0f}")

    # ---- selection-model coefficients
    sel = res_sel["sel"]
    km = q + 2
    g = res_sel["gamma"]                                    # (chains, n, nsel, km+1)
    rows = []
    for ai, j in enumerate(sel):
        name = vit[j]
        m = truth["missingness"][name]
        dx = g[:, :, ai, km] / factor[j]                    # per true SD
        dy = g[:, :, ai, q + 1]
        flat = dx.reshape(-1)
        rows.append(dict(vital=name, mechanism=m["mechanism"], true_gx=m["gx"], gx_mean=flat.mean(),
                         gx_lo=np.quantile(flat, .025), gx_hi=np.quantile(flat, .975),
                         gx_rhat=bm.rhat(dx), true_gy=m["gy"], gy_mean=dy.mean(),
                         gy_lo=np.quantile(dy, .025), gy_hi=np.quantile(dy, .975)))
    gtab = pd.DataFrame(rows)
    gtab.round(3).to_csv(out / "missingness_model.csv", index=False)
    print("\nSelection-model coefficients (probit, per SD of the vital; g_y = effect of the outcome)")
    print(gtab.round(2).to_string(index=False))

    # ---- imputation check
    full_t = full[vit].to_numpy(float).copy()
    for j, nm in enumerate(vit):
        if nm in LOG_VARS:
            full_t[:, j] = np.log(full_t[:, j])
    full_m = (full_t - data["x_loc"]) / data["x_scale"]
    miss = np.isnan(X)
    true_miss_mean = np.array([full_m[miss[:, j], j].mean() for j in range(p)])
    imp = pd.DataFrame({
        "vital": vit, "mech": [truth["missingness"][v]["mechanism"].split(" (")[0].split(" +")[0] for v in vit],
        "observed_mean": 0.0, "true_missing_mean": true_miss_mean,
        "bayes_mar": res_mar["imp_mean"].mean((0, 1)), "bayes_mnar": res_sel["imp_mean"].mean((0, 1)),
    })
    imp.round(3).to_csv(out / "imputation_check.csv", index=False)
    print("\nMean of missing values (observed-data SD units)")
    print(imp.round(2).to_string(index=False))

    make_recovery_figure(
        dict(truth=pd.Series(beta_true), **{k: coef[k] for k in ("Complete-case", "MICE", "Bayes, MAR", "Bayes, MNAR selection")}),
        imp, dict(est=gtab["gx_mean"].values, lo=gtab["gx_lo"].values, hi=gtab["gx_hi"].values, true=gtab["true_gx"].values),
        labels, vit, out / "fig_recovery.png")
    np.savez_compressed(out / "posterior_draws.npz", beta_mar=res_mar["beta"], beta_mnar=res_sel["beta"],
                        gamma_mnar=res_sel["gamma"], labels=np.array(bm.coef_labels(res_sel)))

    # ---- sensitivity analysis
    if not a.skip_sensitivity:
        print("\n== sensitivity analysis (g_x fixed for suspect vitals)", flush=True)
        recs = []
        for s in SENS_GRID:
            t0 = time.time()
            selection = {v: d * s * factor[vit.index(v)] for v, d in SUSPECT.items()}
            r = bm.fit(data, selection, chains=a.chains, n_iter=3000, n_burn=1000, seed=10 + int(s * 100), verbose=False)
            _, tab = bayes_table(r, to_true)
            rec = {"s": s}
            for v in vit:
                rec[f"{v}__est"], rec[f"{v}__lo"], rec[f"{v}__hi"] = tab.loc[labels.index(v), ["est", "lo", "hi"]]
            recs.append(rec)
            print(f"   s={s:.2f}  {time.time() - t0:.0f}s", flush=True)
        sens = pd.DataFrame(recs)
        sens.round(3).to_csv(out / "sensitivity.csv", index=False)
        for v in SUSPECT:
            print(f"   {v:11s} beta by s:", ", ".join(f"{x:.2f}" for x in sens[f"{v}__est"]),
                  f"  (truth {beta_true[labels.index(v)]:.2f})")
        make_sensitivity_figure(sens, {v: beta_true[labels.index(v)] for v in SUSPECT}, out / "fig_sensitivity.png")
        make_recovery_figure(
            dict(truth=pd.Series(beta_true), **{k: coef[k] for k in ("Complete-case", "MICE", "Bayes, MAR", "Bayes, MNAR selection")}),
            imp, dict(est=gtab["gx_mean"].values, lo=gtab["gx_lo"].values, hi=gtab["gx_hi"].values, true=gtab["true_gx"].values),
            labels, vit, out / "fig_recovery.png") 
    print("\ndone ->", out)


if __name__ == "__main__":
    main()


== baselines
   complete cases: 532 of 3000
== Bayesian model, MAR (ignorable)
  [seed 1000] iter 500/6000  4s
  [seed 1000] iter 1000/6000  8s
  [seed 1000] iter 1500/6000  13s
  [seed 1000] iter 2000/6000  17s
  [seed 1000] iter 2500/6000  21s
  [seed 1000] iter 3000/6000  26s
  [seed 1000] iter 3500/6000  30s
  [seed 1000] iter 4000/6000  34s
  [seed 1000] iter 4500/6000  39s
  [seed 1000] iter 5000/6000  43s
  [seed 1000] iter 5500/6000  48s
  [seed 1000] iter 6000/6000  52s
   53s
== Bayesian model, selection model for every vital (g_x free)
  [seed 2000] iter 500/6000  7s
  [seed 2000] iter 1000/6000  14s
  [seed 2000] iter 1500/6000  21s
  [seed 2000] iter 2000/6000  28s
  [seed 2000] iter 2500/6000  35s
  [seed 2000] iter 3000/6000  42s
  [seed 2000] iter 3500/6000  49s
  [seed 2000] iter 4000/6000  56s
  [seed 2000] iter 4500/6000  64s
  [seed 2000] iter 5000/6000  72s
  [seed 2000] iter 5500/6000  80s
  [seed 2000] iter 6000/6000  87s
   88s

Outcome coefficients (log-odds pe

In [8]:
import os
import matplotlib

# 1. Force non-interactive backend (works in terminal, VS Code, WSL, servers)
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
from sklearn.calibration import calibration_curve

# --- Simulate synthetic evaluation data ---
np.random.seed(42)
N = 120

# 15% disease prevalence
y_true = np.random.binomial(1, 0.15, N)

# Missingness indicator: 40% of patients have missing vitals
vitals_missing = np.random.binomial(1, 0.40, N)

# Bayesian Model with Atlas Prior: well-calibrated probabilities
latent_risk = 0.10 + 0.70 * y_true + np.random.normal(0, 0.12, N)
prob_bayes = np.clip(latent_risk, 0.01, 0.98)

# 95% Credible Interval: expands when vitals are missing (epistemic honesty)
base_unc = 0.08
cri_half_width = np.where(vitals_missing == 1, 0.18, base_unc) + np.random.uniform(
    0.01, 0.04, N
)
cri_lower = np.clip(prob_bayes - cri_half_width, 0.0, 1.0)
cri_upper = np.clip(prob_bayes + cri_half_width, 0.0, 1.0)

# Naive complete-case model: overconfident / pushed to extremes
prob_naive = np.clip(
    np.where(prob_bayes > 0.3, prob_bayes**0.55, prob_bayes**1.9), 0.01, 0.99
)

# --- Build the 2 side-by-side plots ---
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 6))

# Panel 1: Calibration / Reliability Curve
fraction_of_pos_bayes, mean_pred_bayes = calibration_curve(
    y_true, prob_bayes, n_bins=6, strategy="quantile"
)
fraction_of_pos_naive, mean_pred_naive = calibration_curve(
    y_true, prob_naive, n_bins=6, strategy="quantile"
)

ax1.plot([0, 1], [0, 1], "k--", label="Perfect Calibration")
ax1.plot(
    mean_pred_bayes,
    fraction_of_pos_bayes,
    "s-",
    color="#1f77b4",
    linewidth=2,
    label="Bayes + Atlas Prior (Calibrated)",
)
ax1.plot(
    mean_pred_naive,
    fraction_of_pos_naive,
    "o-",
    color="#d62728",
    linewidth=2,
    label="Naive Complete-Case (Overconfident)",
)
ax1.set_xlabel("Mean Predicted Probability", fontsize=11)
ax1.set_ylabel("Observed Fraction of Diagnoses", fontsize=11)
ax1.set_title("Plot 1: Calibration / Reliability Curve", fontsize=12, fontweight="bold")
ax1.set_xlim([0, 1])
ax1.set_ylim([0, 1])
ax1.legend(loc="upper left")
ax1.grid(True, linestyle=":", alpha=0.6)

# Panel 2: Posterior Uncertainty vs. Ground Truth
order = np.argsort(prob_bayes)
y_sorted = y_true[order]
p_sorted = prob_bayes[order]
low_sorted = cri_lower[order]
high_sorted = cri_upper[order]
missing_sorted = vitals_missing[order]

for idx in range(N):
    ls = "--" if missing_sorted[idx] == 1 else "-"
    alpha = 0.45 if missing_sorted[idx] == 1 else 0.85
    color = "#d62728" if y_sorted[idx] == 1 else "#2ca02c"

    ax2.plot(
        [idx, idx],
        [low_sorted[idx], high_sorted[idx]],
        linestyle=ls,
        color="gray",
        alpha=alpha,
        linewidth=1.3,
    )
    ax2.scatter(idx, p_sorted[idx], color=color, s=25, zorder=3)

# Legend dummy handles
ax2.scatter([], [], color="#d62728", label="True Diagnosis = 1")
ax2.scatter([], [], color="#2ca02c", label="True Diagnosis = 0")
ax2.plot([], [], "k-", label="Complete Vitals (Narrow CrI)")
ax2.plot([], [], "k--", label="Missing Vitals (Widened CrI)")

ax2.set_xlabel("Patients (Ranked by Predicted Risk)", fontsize=11)
ax2.set_ylabel("Predicted Probability & 95% CrI", fontsize=11)
ax2.set_title(
    "Plot 2: Uncertainty Widening vs. Ground Truth",
    fontsize=12,
    fontweight="bold",
)
ax2.set_ylim([-0.05, 1.05])
ax2.legend(loc="upper left")
ax2.grid(True, linestyle=":", alpha=0.6)

plt.tight_layout()

# Save file directly to disk
output_filename = "evaluation_plots.png"
plt.savefig(output_filename, dpi=300, bbox_inches="tight")
plt.close(fig)

print(f"\n Success! Image saved to: {os.path.abspath(output_filename)}\n")


 Success! Image saved to: /Users/emmamarti/datasci/DataScienceProject/DataanalysProjekt/Preprocessing/evaluation_plots.png



In [2]:
"""ED Atlas lookup and posterior access.

Ships with posterior npz files (small). On first use, computes prediction
grids from the posteriors and caches them in ~/.cache/ed_atlas/.

Shipped data (in ed_atlas/data/):
    atlas_mortality_posterior.npz    — mortality parameter posterior (~300KB)
    atlas_diagnosis_posterior.npz    — all diagnosis posteriors stacked (~300KB per ICD)

Cached grids (in ~/.cache/ed_atlas/, auto-generated):
    atlas_logitnormals.parquet           — diagnosis prediction grid
    atlas_mortality_logitnormals.parquet  — mortality prediction grid

See convert_traces.py for how the npz files are produced from PyMC traces.
"""

from __future__ import annotations

import sys
from dataclasses import dataclass
from pathlib import Path

import numpy as np
import polars as pl

_DATA_DIR = Path(__file__).parent / "data"
_CACHE_DIR = Path.home() / ".cache" / "ed_atlas"

# Age range matching the Bayesian model training grid
AGE_MIN, AGE_MAX = 18, 109
N_AGES = AGE_MAX - AGE_MIN + 1  # 92


@dataclass
class ParameterPosterior:
    """Structured exact posterior for a Bayesian logistic regression model.

    The model formula is:
        logit(P) = beta . x
    where x is built from sex * cr(age) * symptoms (patsy interaction).

    Each prediction uses exactly 20 of the ~4000+ beta dimensions:
    10 "base" dims (intercept, sex, cr[0..3], sex:cr[0..3]) plus
    10 per-symptom dims. The covariance decomposes exactly into:
        base_cov (10x10) + sym_covs[i] (10x10) + cross_covs[i] (10x10)

    Attributes:
        beta_mean: (n_dims,) posterior mean of all betas
        base_cov: (10, 10) covariance of the 10 base dimensions
        sym_covs: (n_symptoms, 10, 10) per-symptom covariance blocks
        cross_covs: (n_symptoms, 10, 10) cross-covariance Cov(base, sym_i)
        sctids: symptom SCTID labels, length n_symptoms
        cr_basis: (92, 4) precomputed natural cubic spline basis for ages 18-109
    """

    beta_mean: np.ndarray
    base_cov: np.ndarray
    sym_covs: np.ndarray
    cross_covs: np.ndarray
    sctids: list[str]
    cr_basis: np.ndarray

    @staticmethod
    def load(path: Path) -> ParameterPosterior:
        """Load from an npz file produced by convert_traces.py."""
        d = np.load(path)
        return ParameterPosterior(
            beta_mean=d["beta_mean"],
            base_cov=d["base_cov"],
            sym_covs=d["sym_covs"],
            cross_covs=d["cross_covs"],
            sctids=d["sctids"].tolist(),
            cr_basis=d["cr_basis"],
        )

    @property
    def n_symptoms(self) -> int:
        return len(self.sctids)

    def _sctid_index(self, sctid: str) -> int:
        try:
            return self.sctids.index(sctid)
        except ValueError:
            raise KeyError(f"Unknown sctid {sctid!r}. Available: {len(self.sctids)}")

    def _symptom_dims(self, si: int) -> list[int]:
        """Return the 10 beta dimensions for symptom index si.

        Matches the patsy column layout for the interaction formula.
        """
        S = self.n_symptoms
        return [
            10 + si,                       # main effect
            10 + S + si,                   # sex:symptom
            10 + 2 * S + 4 * si + 0,      # cr[0]:symptom
            10 + 2 * S + 4 * si + 1,      # cr[1]:symptom
            10 + 2 * S + 4 * si + 2,      # cr[2]:symptom
            10 + 2 * S + 4 * si + 3,      # cr[3]:symptom
            10 + 6 * S + 4 * si + 0,      # sex:cr[0]:symptom
            10 + 6 * S + 4 * si + 1,      # sex:cr[1]:symptom
            10 + 6 * S + 4 * si + 2,      # sex:cr[2]:symptom
            10 + 6 * S + 4 * si + 3,      # sex:cr[3]:symptom
        ]

    def _design_vector(self, age: int, is_male: int) -> tuple[np.ndarray, np.ndarray]:
        """Build the 20-dim design vector for a single prediction.

        Returns (x_base, x_sym) each length 10.
        """
        cr = self.cr_basis[age - AGE_MIN]  # (4,) from precomputed basis
        sex = float(is_male)

        # Base: [intercept, sex, cr[0..3], sex*cr[0..3]]
        x_base = np.array(
            [1.0, sex, cr[0], cr[1], cr[2], cr[3],
             sex * cr[0], sex * cr[1], sex * cr[2], sex * cr[3]],
            dtype=np.float32,
        )

        # Symptom: [1, sex, cr[0..3], sex*cr[0..3]] (same pattern, symptom=1)
        x_sym = np.array(
            [1.0, sex, cr[0], cr[1], cr[2], cr[3],
             sex * cr[0], sex * cr[1], sex * cr[2], sex * cr[3]],
            dtype=np.float32,
        )

        return x_base, x_sym

    def predict(self, age: int, is_male: int, sctid: str) -> tuple[float, float]:
        """Compute posterior predictive LogitNormal(mu, sigma) from parameters.

        Returns (mu, sigma) in logit space. Apply sigmoid(mu) for point estimate.
        """
        si = self._sctid_index(sctid)
        x_base, x_sym = self._design_vector(age, is_male)

        # Mean: beta_mean[base_dims] . x_base + beta_mean[sym_dims] . x_sym
        base_dims = list(range(10))
        sym_dims = self._symptom_dims(si)
        mu = (
            float(self.beta_mean[base_dims] @ x_base)
            + float(self.beta_mean[sym_dims] @ x_sym)
        )

        # Variance: exact decomposition
        var = (
            float(x_base @ self.base_cov @ x_base)
            + float(x_sym @ self.sym_covs[si] @ x_sym)
            + 2.0 * float(x_base @ self.cross_covs[si] @ x_sym)
        )
        sigma = float(np.sqrt(max(var, 0.0)))

        return mu, sigma

    def beta_summary(self, sctid: str) -> tuple[np.ndarray, np.ndarray]:
        """Return (mean, std) of the 20 active betas for a symptom.

        First 10 are base dims, last 10 are symptom dims.
        """
        si = self._sctid_index(sctid)
        base_dims = list(range(10))
        sym_dims = self._symptom_dims(si)
        dims = base_dims + sym_dims

        mean = self.beta_mean[dims]

        # Diagonal of the 20x20 joint covariance
        base_var = np.diag(self.base_cov)
        sym_var = np.diag(self.sym_covs[si])
        std = np.sqrt(np.concatenate([base_var, sym_var]))

        return mean, std


# ── Grid generation from posteriors ──────────────────────────────────


def _build_design_matrices(cr_basis: np.ndarray) -> tuple[np.ndarray, np.ndarray]:
    """Build design matrices for all (age, sex) combinations.

    Returns (X_base, X_sym) each shape (184, 10) for 92 ages × 2 sexes.
    Row order: age=18 sex=0, age=18 sex=1, age=19 sex=0, ...
    """
    n_rows = N_AGES * 2
    X_base = np.empty((n_rows, 10), dtype=np.float32)
    X_sym = np.empty((n_rows, 10), dtype=np.float32)

    for ai in range(N_AGES):
        cr = cr_basis[ai]
        for sx in range(2):
            row = ai * 2 + sx
            s = float(sx)
            v = [1.0, s, cr[0], cr[1], cr[2], cr[3],
                 s * cr[0], s * cr[1], s * cr[2], s * cr[3]]
            X_base[row] = v
            X_sym[row] = v

    return X_base, X_sym


def _generate_mortality_grid(post: ParameterPosterior) -> pl.DataFrame:
    """Compute mortality prediction grid from posterior parameters."""
    X_base, X_sym = _build_design_matrices(post.cr_basis)
    n_points = X_base.shape[0]  # 184

    # Precompute base quadratic form (shared across symptoms)
    base_quad = (X_base @ post.base_cov * X_base).sum(axis=1)  # (184,)

    sctid_col = []
    age_col = []
    sex_col = []
    mu_col = []
    sigma_col = []

    beta_base = post.beta_mean[:10]

    for si in range(post.n_symptoms):
        sym_dims = post._symptom_dims(si)
        beta_sym = post.beta_mean[sym_dims]

        mu = X_base @ beta_base + X_sym @ beta_sym  # (184,)
        sym_quad = (X_sym @ post.sym_covs[si] * X_sym).sum(axis=1)
        cross_quad = (X_base @ post.cross_covs[si] * X_sym).sum(axis=1)
        var = base_quad + sym_quad + 2.0 * cross_quad
        sigma = np.sqrt(np.maximum(var, 0.0))

        for row in range(n_points):
            ai, sx = divmod(row, 2)
            sctid_col.append(post.sctids[si])
            age_col.append(AGE_MIN + ai)
            sex_col.append(sx)
            mu_col.append(float(mu[row]))
            sigma_col.append(float(sigma[row]))

    return pl.DataFrame({
        "sctid": sctid_col,
        "age": age_col,
        "is_male": sex_col,
        "mean_prediction": np.array(mu_col, dtype=np.float32),
        "std_prediction": np.array(sigma_col, dtype=np.float32),
    }).sort(["sctid", "age", "is_male"])


def _generate_diagnosis_grid(
    diag_npz_path: Path,
) -> pl.DataFrame:
    """Compute diagnosis prediction grid from stacked posterior npz."""
    d = np.load(diag_npz_path)
    icd_codes = d["icd_codes"].tolist()
    sctids = d["sctids"].tolist()
    cr_basis = d["cr_basis"]
    n_icds = len(icd_codes)
    n_symptoms = len(sctids)

    X_base, X_sym = _build_design_matrices(cr_basis)
    n_points = X_base.shape[0]  # 184

    total_rows = n_icds * n_symptoms * n_points
    all_sctid = []
    all_age = np.empty(total_rows, dtype=np.int64)
    all_sex = np.empty(total_rows, dtype=np.int64)
    all_icd = []
    all_mu = np.empty(total_rows, dtype=np.float32)
    all_sigma = np.empty(total_rows, dtype=np.float32)

    # Precompute age/sex pattern (repeats for every symptom and ICD)
    ages_184 = np.array([AGE_MIN + ai for ai in range(N_AGES) for _ in range(2)], dtype=np.int64)
    sexes_184 = np.array([sx for _ in range(N_AGES) for sx in range(2)], dtype=np.int64)

    row_offset = 0
    for ki in range(n_icds):
        icd = icd_codes[ki]
        beta_mean = d["beta_means"][ki]
        base_cov = d["base_covs"][ki]
        sym_covs_k = d["sym_covs"][ki]
        cross_covs_k = d["cross_covs"][ki]

        beta_base = beta_mean[:10]
        base_quad = (X_base @ base_cov * X_base).sum(axis=1)

        for si in range(n_symptoms):
            S = n_symptoms
            sym_dims = [
                10 + si, 10 + S + si,
                10 + 2*S + 4*si, 10 + 2*S + 4*si+1, 10 + 2*S + 4*si+2, 10 + 2*S + 4*si+3,
                10 + 6*S + 4*si, 10 + 6*S + 4*si+1, 10 + 6*S + 4*si+2, 10 + 6*S + 4*si+3,
            ]
            beta_sym = beta_mean[sym_dims]

            mu = X_base @ beta_base + X_sym @ beta_sym
            sym_quad = (X_sym @ sym_covs_k[si] * X_sym).sum(axis=1)
            cross_quad = (X_base @ cross_covs_k[si] * X_sym).sum(axis=1)
            var = base_quad + sym_quad + 2.0 * cross_quad
            sigma = np.sqrt(np.maximum(var, 0.0))

            end = row_offset + n_points
            all_sctid.extend([sctids[si]] * n_points)
            all_icd.extend([icd] * n_points)
            all_age[row_offset:end] = ages_184
            all_sex[row_offset:end] = sexes_184
            all_mu[row_offset:end] = mu
            all_sigma[row_offset:end] = sigma
            row_offset = end

        if (ki + 1) % 50 == 0 or ki == n_icds - 1:
            print(f"    {ki+1}/{n_icds} ICD codes", file=sys.stderr)

    return pl.DataFrame({
        "age": all_age[:row_offset],
        "is_male": all_sex[:row_offset],
        "sctid": all_sctid,
        "mean_prediction": all_mu[:row_offset],
        "std_prediction": all_sigma[:row_offset],
        "ICD code": all_icd,
    })


def _ensure_cache() -> Path:
    """Ensure cached grids exist. Generate from posteriors if needed.

    Returns the cache directory path.
    """
    _CACHE_DIR.mkdir(parents=True, exist_ok=True)

    mort_grid = _CACHE_DIR / "atlas_mortality_logitnormals.parquet"
    diag_grid = _CACHE_DIR / "atlas_logitnormals.parquet"

    mort_npz = _DATA_DIR / "atlas_mortality_posterior.npz"
    diag_npz = _DATA_DIR / "atlas_diagnosis_posterior.npz"

    if mort_grid.exists() and diag_grid.exists():
        return _CACHE_DIR

    if not mort_npz.exists():
        raise FileNotFoundError(
            f"Posterior data not found: {mort_npz}\n"
            f"Place atlas_mortality_posterior.npz in {_DATA_DIR}/"
        )

    print("╭─ ED Atlas ─────────────────────────────────────╮", file=sys.stderr)
    print("│ Populating prediction cache from posteriors...  │", file=sys.stderr)
    print("│ This only happens once.                         │", file=sys.stderr)
    print("╰─────────────────────────────────────────────────╯", file=sys.stderr)

    if not mort_grid.exists():
        print("  Generating mortality grid...", file=sys.stderr)
        post = ParameterPosterior.load(mort_npz)
        df = _generate_mortality_grid(post)
        df.write_parquet(mort_grid)
        print(f"  Saved {mort_grid} ({df.shape[0]} rows)", file=sys.stderr)

    if not diag_grid.exists() and diag_npz.exists():
        print("  Generating diagnosis grid...", file=sys.stderr)
        df = _generate_diagnosis_grid(diag_npz)
        df.write_parquet(diag_grid)
        print(f"  Saved {diag_grid} ({df.shape[0]} rows)", file=sys.stderr)

    print("  Cache ready.", file=sys.stderr)
    return _CACHE_DIR


class EdAtlas:
    """ED Atlas: Bayesian logistic regression priors for ED triage.

    Provides precomputed LogitNormal predictive distributions for patients
    presenting to the emergency department, conditioned on age, sex, and
    presenting complaint (SNOMED CT concept ID).

    Two outcome types:
      - Diagnosis (ICD-10 codes within 30 days)
      - Death (30-day mortality)

    Three levels of access:
      1. predict() / predict_death() — posterior predictive mean (fast lookup)
      2. predict_distribution() / predict_death_distribution() — LogitNormal params
      3. mortality_posterior / diagnosis_posterior() — full parameter posterior

    When a patient has two complaints, predictions mix in probability space:
        P = 0.5 * E[sigmoid(z_1)] + 0.5 * E[sigmoid(z_2)],  z_i ~ N(mu_i, sigma_i^2)
    This is the correct E[P] under complaint uncertainty.
    """

    def __init__(self, resources_dir: Path | None = None) -> None:
        res = resources_dir or _ensure_cache()

        # --- Diagnosis grid ---
        diag_path = res / "atlas_logitnormals.parquet"
        if not diag_path.exists():
            raise FileNotFoundError(f"Diagnosis grid not found: {diag_path}")

        diag_df = pl.read_parquet(diag_path)
        self._icd_codes = sorted(diag_df["ICD code"].unique().to_list())
        self._diag_sctids = sorted(diag_df["sctid"].unique().to_list())
        n_sctids_d = len(self._diag_sctids)
        n_icds = len(self._icd_codes)

        # Build 4D lookup: (sctid_idx, age_idx, sex, icd_idx)
        sctid_to_idx_d = {s: i for i, s in enumerate(self._diag_sctids)}
        icd_to_idx = {c: i for i, c in enumerate(self._icd_codes)}

        self._diag_mus = np.full((n_sctids_d, N_AGES, 2, n_icds), np.nan, dtype=np.float32)
        self._diag_sigmas = np.full((n_sctids_d, N_AGES, 2, n_icds), np.nan, dtype=np.float32)

        # Vectorized fill
        si = diag_df["sctid"].replace_strict(sctid_to_idx_d, return_dtype=pl.Int64).to_numpy()
        ai = (diag_df["age"].to_numpy() - AGE_MIN)
        xi = diag_df["is_male"].to_numpy()
        ii = diag_df["ICD code"].replace_strict(icd_to_idx, return_dtype=pl.Int64).to_numpy()
        self._diag_mus[si, ai, xi, ii] = diag_df["mean_prediction"].to_numpy()
        self._diag_sigmas[si, ai, xi, ii] = diag_df["std_prediction"].to_numpy()

        assert not np.any(np.isnan(self._diag_mus)), "Missing cells in diagnosis grid"

        self._diag_sctid_to_idx = sctid_to_idx_d
        self._icd_to_idx = icd_to_idx

        # --- Mortality grid ---
        mort_path = res / "atlas_mortality_logitnormals.parquet"
        if not mort_path.exists():
            raise FileNotFoundError(f"Mortality grid not found: {mort_path}")

        mort_df = pl.read_parquet(mort_path)
        self._mort_sctids = sorted(mort_df["sctid"].unique().to_list())
        n_sctids_m = len(self._mort_sctids)
        sctid_to_idx_m = {s: i for i, s in enumerate(self._mort_sctids)}

        self._mort_mus = np.full((n_sctids_m, N_AGES, 2), np.nan, dtype=np.float32)
        self._mort_sigmas = np.full((n_sctids_m, N_AGES, 2), np.nan, dtype=np.float32)

        si_m = mort_df["sctid"].replace_strict(sctid_to_idx_m, return_dtype=pl.Int64).to_numpy()
        ai_m = (mort_df["age"].to_numpy() - AGE_MIN)
        xi_m = mort_df["is_male"].to_numpy()
        self._mort_mus[si_m, ai_m, xi_m] = mort_df["mean_prediction"].to_numpy()
        self._mort_sigmas[si_m, ai_m, xi_m] = mort_df["std_prediction"].to_numpy()

        assert not np.any(np.isnan(self._mort_mus)), "Missing cells in mortality grid"

        self._mort_sctid_to_idx = sctid_to_idx_m

        # --- Posteriors (loaded lazily from shipped data) ---
        self._mort_posterior: ParameterPosterior | None = None
        self._diag_posteriors: dict[str, ParameterPosterior] = {}

    def __repr__(self) -> str:
        return (
            f"EdAtlas(diag_sctids={len(self._diag_sctids)}, "
            f"mort_sctids={len(self._mort_sctids)}, "
            f"icd_codes={len(self._icd_codes)}, "
            f"ages={AGE_MIN}-{AGE_MAX})"
        )

    @property
    def icd_codes(self) -> list[str]:
        return self._icd_codes

    @property
    def diag_sctids(self) -> list[str]:
        return self._diag_sctids

    @property
    def mort_sctids(self) -> list[str]:
        return self._mort_sctids

    # ── Diagnosis prediction ──────────────────────────────────────────

    def _validate_age(self, age: int) -> int:
        if not (AGE_MIN <= age <= AGE_MAX):
            raise ValueError(f"Age {age} outside range [{AGE_MIN}, {AGE_MAX}]")
        return age - AGE_MIN

    def predict(
        self, age: int, is_male: int, sctids: list[str]
    ) -> np.ndarray:
        """Posterior predictive mean E[sigmoid(logit)] per ICD code (quadrature).

        Args:
            age: Patient age (18-109).
            is_male: 0 or 1.
            sctids: 1 or 2 presenting complaint SCTID strings.

        Returns:
            (n_icds,) float32 array of probabilities.
        """
        ai = self._validate_age(age)
        if len(sctids) == 1:
            si = self._diag_sctid_to_idx[sctids[0]]
            return _logitnormal_mean(
                self._diag_mus[si, ai, is_male], self._diag_sigmas[si, ai, is_male]
            ).astype(np.float32)
        elif len(sctids) == 2:
            si0 = self._diag_sctid_to_idx[sctids[0]]
            si1 = self._diag_sctid_to_idx[sctids[1]]
            p0 = _logitnormal_mean(
                self._diag_mus[si0, ai, is_male], self._diag_sigmas[si0, ai, is_male]
            )
            p1 = _logitnormal_mean(
                self._diag_mus[si1, ai, is_male], self._diag_sigmas[si1, ai, is_male]
            )
            return (0.5 * p0 + 0.5 * p1).astype(np.float32)
        else:
            raise ValueError(f"Expected 1 or 2 sctids, got {len(sctids)}")

    def predict_batch(
        self,
        ages: np.ndarray,
        is_males: np.ndarray,
        sctids_list: list[list[str]],
    ) -> np.ndarray:
        """Batch diagnosis prediction. Returns (N, n_icds) float32."""
        N = len(ages)
        out = np.empty((N, len(self._icd_codes)), dtype=np.float32)
        for i in range(N):
            out[i] = self.predict(int(ages[i]), int(is_males[i]), sctids_list[i])
        return out

    def predict_distribution(
        self, age: int, is_male: int, sctids: list[str]
    ) -> tuple[np.ndarray, np.ndarray] | tuple[tuple[np.ndarray, np.ndarray], tuple[np.ndarray, np.ndarray]]:
        """LogitNormal(mu, sigma) parameters per ICD code.

        For 1 complaint: returns (mu, sigma) each (n_icds,).
        For 2 complaints: returns ((mu1, sigma1), (mu2, sigma2)).
        """
        ai = self._validate_age(age)
        if len(sctids) == 1:
            si = self._diag_sctid_to_idx[sctids[0]]
            return self._diag_mus[si, ai, is_male].copy(), self._diag_sigmas[si, ai, is_male].copy()
        elif len(sctids) == 2:
            si0 = self._diag_sctid_to_idx[sctids[0]]
            si1 = self._diag_sctid_to_idx[sctids[1]]
            return (
                (self._diag_mus[si0, ai, is_male].copy(), self._diag_sigmas[si0, ai, is_male].copy()),
                (self._diag_mus[si1, ai, is_male].copy(), self._diag_sigmas[si1, ai, is_male].copy()),
            )
        else:
            raise ValueError(f"Expected 1 or 2 sctids, got {len(sctids)}")

    # ── Mortality prediction ──────────────────────────────────────────

    def predict_death(
        self, age: int, is_male: int, sctids: list[str]
    ) -> float:
        """Posterior predictive mean E[P(death within 30 days)] (quadrature).

        Args:
            age: Patient age (18-109).
            is_male: 0 or 1.
            sctids: 1 or 2 presenting complaint SCTID strings.

        Returns:
            Probability as float.
        """
        ai = self._validate_age(age)
        if len(sctids) == 1:
            si = self._mort_sctid_to_idx[sctids[0]]
            return float(_logitnormal_mean(
                self._mort_mus[si, ai, is_male], self._mort_sigmas[si, ai, is_male]
            ))
        elif len(sctids) == 2:
            si0 = self._mort_sctid_to_idx[sctids[0]]
            si1 = self._mort_sctid_to_idx[sctids[1]]
            p0 = _logitnormal_mean(
                self._mort_mus[si0, ai, is_male], self._mort_sigmas[si0, ai, is_male]
            )
            p1 = _logitnormal_mean(
                self._mort_mus[si1, ai, is_male], self._mort_sigmas[si1, ai, is_male]
            )
            return float(0.5 * p0 + 0.5 * p1)
        else:
            raise ValueError(f"Expected 1 or 2 sctids, got {len(sctids)}")

    def predict_death_batch(
        self,
        ages: np.ndarray,
        is_males: np.ndarray,
        sctids_list: list[list[str]],
    ) -> np.ndarray:
        """Batch mortality prediction. Returns (N,) float32."""
        N = len(ages)
        out = np.empty(N, dtype=np.float32)
        for i in range(N):
            out[i] = self.predict_death(int(ages[i]), int(is_males[i]), sctids_list[i])
        return out

    def predict_death_distribution(
        self, age: int, is_male: int, sctids: list[str]
    ) -> tuple[float, float] | tuple[tuple[float, float], tuple[float, float]]:
        """LogitNormal(mu, sigma) for mortality.

        For 1 complaint: returns (mu, sigma) scalars.
        For 2 complaints: returns ((mu1, sigma1), (mu2, sigma2)).
        """
        ai = self._validate_age(age)
        if len(sctids) == 1:
            si = self._mort_sctid_to_idx[sctids[0]]
            return float(self._mort_mus[si, ai, is_male]), float(self._mort_sigmas[si, ai, is_male])
        elif len(sctids) == 2:
            si0 = self._mort_sctid_to_idx[sctids[0]]
            si1 = self._mort_sctid_to_idx[sctids[1]]
            return (
                (float(self._mort_mus[si0, ai, is_male]), float(self._mort_sigmas[si0, ai, is_male])),
                (float(self._mort_mus[si1, ai, is_male]), float(self._mort_sigmas[si1, ai, is_male])),
            )
        else:
            raise ValueError(f"Expected 1 or 2 sctids, got {len(sctids)}")

    # ── Parameter posteriors ──────────────────────────────────────────

    @property
    def mortality_posterior(self) -> ParameterPosterior:
        """Structured exact posterior for the mortality model."""
        if self._mort_posterior is None:
            path = _DATA_DIR / "atlas_mortality_posterior.npz"
            if not path.exists():
                raise FileNotFoundError(
                    f"Mortality posterior not found: {path}. "
                    f"Place atlas_mortality_posterior.npz in {_DATA_DIR}/"
                )
            self._mort_posterior = ParameterPosterior.load(path)
        return self._mort_posterior

    def diagnosis_posterior(self, icd_code: str) -> ParameterPosterior:
        """Structured exact posterior for a diagnosis model.

        Args:
            icd_code: ICD-10 code (e.g. "X80").
        """
        if icd_code not in self._diag_posteriors:
            self._load_diagnosis_posteriors()
            if icd_code not in self._diag_posteriors:
                raise KeyError(
                    f"No posterior for ICD code {icd_code!r}. "
                    f"Available: {sorted(self._diag_posteriors.keys())}"
                )
        return self._diag_posteriors[icd_code]

    def _load_diagnosis_posteriors(self) -> None:
        """Load all diagnosis posteriors from single stacked npz."""
        if self._diag_posteriors:
            return
        path = _DATA_DIR / "atlas_diagnosis_posterior.npz"
        if not path.exists():
            raise FileNotFoundError(
                f"Diagnosis posteriors not found: {path}. "
                f"Place atlas_diagnosis_posterior.npz in {_DATA_DIR}/"
            )
        d = np.load(path)
        icd_codes = d["icd_codes"].tolist()
        sctids = d["sctids"].tolist()
        cr_basis = d["cr_basis"]
        for k, icd in enumerate(icd_codes):
            self._diag_posteriors[icd] = ParameterPosterior(
                beta_mean=d["beta_means"][k],
                base_cov=d["base_covs"][k],
                sym_covs=d["sym_covs"][k],
                cross_covs=d["cross_covs"][k],
                sctids=sctids,
                cr_basis=cr_basis,
            )

    def available_posteriors(self) -> dict[str, list[str]]:
        """List available posterior files."""
        result: dict[str, list[str]] = {}
        mort_path = _DATA_DIR / "atlas_mortality_posterior.npz"
        if mort_path.exists():
            result["mortality"] = ["mortality"]
        diag_path = _DATA_DIR / "atlas_diagnosis_posterior.npz"
        if diag_path.exists():
            result["diagnosis"] = np.load(diag_path)["icd_codes"].tolist()
        return result


def _sigmoid(x: np.ndarray | float) -> np.ndarray | float:
    return 1.0 / (1.0 + np.exp(-x))


# Probabilists' Gauss-Hermite nodes/weights: E[f(Z)] ~ sum_i w_i f(x_i), Z ~ N(0, 1).
_GH_NODES, _GH_WEIGHTS = np.polynomial.hermite_e.hermegauss(64)
_GH_WEIGHTS = _GH_WEIGHTS / _GH_WEIGHTS.sum()


def _logitnormal_mean(mu: np.ndarray | float, sigma: np.ndarray | float) -> np.ndarray | float:
    """E[sigmoid(Z)], Z ~ N(mu, sigma^2), by Gauss-Hermite quadrature.

    Computed in probability space; sigmoid(mu) is the median, not the mean,
    and differs substantially from it when sigma is large.
    """
    mu = np.asarray(mu, dtype=np.float64)
    sigma = np.asarray(sigma, dtype=np.float64)
    z = mu[..., None] + sigma[..., None] * _GH_NODES
    return _sigmoid(z) @ _GH_WEIGHTS


ModuleNotFoundError: No module named 'polars'